In [2]:
#Boilerplate setup, common across all notebooks
from pathlib import Path
from graphviz import Digraph
from collections import Counter
import sys
import pandas as pd
import numpy as np
import time
import networkx as nx
import matplotlib.pyplot as plt
import itertools
import random

## import causal discovery algorithm code here ##

#set project root as .../recidivism-causal
project_root = Path("/dcs/23/u2200504/thesis/recidivism-causal").resolve()

#set causal pitfalls root
cp_root = project_root / "data" / "raw" / "CausalPitfallsData"

#set NIJ root
nij_root = project_root / "data" / "processed"

#extension path as req.
output_dir = project_root/"results"/"graphs_fci"
output_dir.mkdir(parents=True,exist_ok=True)

In [36]:
# Utility Functions

#scoring utility function given two DIRECTED adjacency matricies
'''
IN: W_est: Estimated adjacency matrix,
    W_true: Ground truth adjacency matrix,
    labels_est: node labels of estimated adjacency matrix,
    labels_true: true node labels
OUT: Dictionary containing, FP, FN, TN, SHD, TPR, FPR, FDR
'''
def score_graph(W_est, W_true, labels_est, labels_true):
    #number of variables
    p = W_est.shape[0]

    #node orderings
    labels_est = list(labels_est)
    labels_true = list(labels_true)
    common = sorted(set(labels_est) & set(labels_true))

    #map ids
    idx_est = [labels_est.index(v) for v in common]
    idx_true = [labels_true.index(v) for v in common]

    W_est_aligned = np.asarray(W_est)[np.ix_(idx_est, idx_est)]
    W_true_aligned = np.asarray(W_true)[np.ix_(idx_true, idx_true)]

    est = (W_est!=0).astype(int)
    true = (W_true !=0).astype(int)

    #true positive, false positive, false negative, true negative
    tp = np.sum((est==1) & (true==1))
    fp = np.sum((est==1) & (true == 0))
    fn = np.sum((est==0) & (true == 1))
    tn = np.sum((est==0) & (true == 0))

    #structural hamming distance, true positive rate, false positive rate
    shd = fp + fn
    tpr = tp/(tp+fn) if (tp+fn) > 0 else np.nan
    fpr = fp/(fp+tn) if (fp + tn)>0 else np.nan
    fdr = fp/(tp+fp) if (tp+fp)>0 else np.nan

    #store scores in dictionary format for each 'experiment'
    return dict(TP=int(tp), FP=int(fp), FN=int(fn), TN=int(tn), SHD=int(shd), TPR = tpr, FPR = fpr, FDR= fdr)

#Function to encode non numeric values using their category codes
"""
IN: df: Dataframe containing data to be encoded
OUT: df_enc: encoded data in a Dataframe
"""
def encode_mixed_df(df):
    df_enc = df.copy()
    for col in df_enc.columns:
        #use categorical code for non numeric
        if not np.issubdtype(df_enc[col].dtype, np.number):
            df_enc[col] = df_enc[col].astype("category").cat.codes
    return df_enc

#Function to draw graph from nx.Digraph object
"""
IN: G: nx.Digraph object encoding graph to be drawn
    out_path: path to store graph
OUT: None. saves graph drawing
"""
def draw_graphviz_from_nx(G, out_path, engine="dot"):
    node_labels = list(G.nodes())
    #save the graph as a png
    g = Digraph(format="png", engine=engine)
    
    g.attr(rankdir="TB")
    g.attr("node", shape="ellipse", style="solid",
           color="black", fontname="Helvetica", fontsize="10")
    g.attr("edge", color="black", arrowsize="0.7")
    
    for n in node_labels:
        g.node(n, label=n)
    
    for u, v in G.edges():
        g.edge(u, v)

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    g.render(filename=out_path.with_suffix("").as_posix(), cleanup=True)

#Function to compute the incompatibility score of an adjacency matrix
"""
IN: W_full: adjacency matrix of the estimated causal graph
    k: number of runs, we set this to 5 as default
    n_subsets: number of subsets to sample
    seed: random seed
OUT: avg SHD value across k runs := incompatibility score
"""
def incompatibility_score(W_full, k=5, n_subsets=50, seed=0):
    #sample random k-node subsets, run the algorithm again, and compare compatibility with learned DAG
    rng = random.Random(seed)
    W_full = (np.asarray(W_full) != 0).astype(int)
    d = W_full.shape[0]

    if k > d:
        raise ValueError("Subset size k cannot exceed number of variables")

    #computes SHD between two adjacency matrices
    def shd(A, B):
        A = (A != 0).astype(int)
        B = (B != 0).astype(int)
        return np.sum(A != B)

    shd_vals = []

    for _ in range(n_subsets):
        #randomly sample a k-variable subset of all variables
        idx = sorted(rng.sample(range(d), k))

        # 1)restrict the full graph to this subset
        W_restricted = W_full[np.ix_(idx, idx)]

        # 2)run cd algorithm on subset of variables
        csv_path= nij_root/"NIJ_lean_compact_onehot.csv"
        df = pd.read_csv(csv_path)
        df_subset = df.iloc[:, idx]
        W_subset = run_dagslam(df_subset);
 
        # 3) compute SHD between the restricted full DAG and the subset DAG
        shd_vals.append(shd(W_restricted, W_subset))

    # incompatibility score ~= average SHD across subsets
    return np.mean(shd_vals)

#function to interpret incompatibility score as a % of disagreeing edges
"""
IN: incompat_score: incompat. score to interpret
    k: size of subset corresponding to incompatability score
OUT: disagreeing edges as a %
"""
def goodness(incompat_score, k):
    poss_edges = k*(k-1)
    frac = incompat_score/poss_edges
    return(frac*100)

#function to run edge stability experiment on a causal discovery algorithm
"""
IN: df: Dataframe containing original data
    B: number of runs
    seed: seed to generate random numbers from
OUT: edge_freq: np array containing proportion of runs where edge appears
"""
def bootstrap_edge_stability(df, B=20, seed=0):
    rng = np.random.default_rng(seed)
    edge_counts = np.zeros((d, d), dtype=int)

    for b in range(B):
        #sample B rows with replacement from original learning data
        idx = rng.integers(low=0, high=len(df), size=len(df))
        df_boot = df.iloc[idx, :]

        #run DAGSLAM on each bootstrap sample
        W_est = run_dagslam(df_boot)           # shape (d, d), aligned with columns
        W_bin = (np.asarray(W_est) != 0).astype(int)

        #accumulate edge counts
        edge_counts += W_bin

    #convert to percentage appearances
    edge_freq = edge_counts / B
    return edge_freq

In [5]:
#FCI-specific utility functions

#Scoring utility function for a PDAG adjacency matrix and a directed adjacency ground truth
'''
IN: W_est: Estimated PDAG adjacency matrix, (encoded with -1, 1 or 2)
    W_true: Ground truth adjacency matrix,
    labels_est: node labels of estimated adjacency matrix,
    labels_true: true node labels
OUT: Two separate dictionaries containing, FP, FN, TN, SHD, TPR, FPR, FDR
    skeleton metric scores with no direction penalty/reward
    oriented mmetric scores with direciton, no orientation i.e (o) mark counts as a penalty
'''
#pdag encoding -1 -> arrowtail, 1 -> arrowhead, 2 -> circle mark
def score_pdag(W_pdag, W_true, labels_pdag, labels_true):
    W_pdag = np.asarray(W_pdag)
    W_true = np.asarray(W_true)

    #skeleton graphs ignore directionality, score presence of edge
    S_est = ((W_pdag != 0) | (W_pdag.T != 0)).astype(int)
    S_true = ((W_true != 0) | (W_true.T != 0)).astype(int)

    s_metrics = score_graph(S_est,S_true, labels_pdag, labels_true)

    #orientation metric, score directed edges    
    #node orderings
    labels_pdag = list(labels_pdag)
    labels_true = list(labels_true)
    common = sorted(set(labels_pdag) & set(labels_true))

    #map ids
    idx_est = [labels_pdag.index(v) for v in common]
    idx_true = [labels_true.index(v) for v in common]

    W_pdag = np.asarray(W_pdag)[np.ix_(idx_est, idx_est)]
    W_true = np.asarray(W_true)[np.ix_(idx_true, idx_true)]

    #number of variables
    p = W_true.shape[0]
    tp_dir = fp_dir=fn_dir = tn_dir=0

    for i in range(p):
        for j in range(p):
            if i ==j:
                #print("i was equal j")
                continue

            true_ij = W_true[i,j]
            true_ji = W_true[j,i]
            est_ij = W_pdag[i,j]
            est_ji= W_pdag[j,i]

            #consider only pairs that are adjacent in ground truth DAG
            if(true_ij==0) and (true_ji==0):
                #print("wasnt adjacent in gt")
                continue

            # set correct direction according to the ground truth
            if true_ij == 1:
                true_dir = (i,j)
            else:
                true_dir = (j,i)

            #score direction according to the directed edge in the PDAG
            if is_dir(W_pdag, i, j):
                est_dir = (i, j)
            elif is_dir(W_pdag, j, i):
                est_dir = (j, i)
            elif (W_pdag[i, j] != 0) or (W_pdag[j, i] != 0):
                est_dir = None   # edge not oriented
            else:
                est_dir = None   # no edge

            if est_dir is None:
                # adjacency handled by skeleton metrics
                fn_dir += 1   # methodological choice: can choose to count lack of orientation as FN..
            elif est_dir == true_dir:
                tp_dir += 1
            else:
                fp_dir += 1

    shd_dir = fp_dir + fn_dir
    tpr_dir = tp_dir/(tp_dir+fn_dir) if (tp_dir+fn_dir) > 0 else np.nan
    fpr_dir = fp_dir/(fp_dir+tn_dir) if (fp_dir+tn_dir) > 0 else np.nan
    fdr_dir = fp_dir/(tp_dir+fp_dir) if (tp_dir+fp_dir) > 0 else np.nan

    orient_metrics = dict(
        TP=int(tp_dir), FP=int(fp_dir), FN=int(fn_dir), TN=int(tn_dir),
        SHD=int(shd_dir), TPR=tpr_dir, FPR=fpr_dir, FDR=fdr_dir
    )

    #return metric dictionaries, prefix with skeleton or orientation
    return {
        **{f"skel_{k}": v for k, v in s_metrics.items()},
        **{f"orient_{k}": v for k, v in orient_metrics.items()},
    }

#function to determine orientation of edge between two nodes in a graph
"""
IN: W: Adjacency matrix of a directed graph
    i: id of the first node
    j: id of the second/target node
OUT: TRUE if i -> j else FALSE
"""
def is_dir(W,i,j):
    # an edge is oriented i -> j if j has arrowhead (enc as 1) and i doesn't (anything but 1)
    return (W[i, j] != 1) and (W[j, i] == 1)

In [3]:
#Utility functions for DAGSLAM

#add dagslam implementation to system path
dagslam_path = project_root/"code"/"dagslam"/"DAGSLAM Causal Bayesian Network Structure Learning of Mixed Type Data"/"_dagslam"
sys.path.append(str(dagslam_path))
import importlib
import DAGSLAM 
importlib.reload(DAGSLAM)
from DAGSLAM import dagslam

#uses DAGSLAM implementation code authored by Yuanyuan Zhao. 
#https://github.com/yuanyuan-zhao-pku/DAGSLAM/blob/main/DAGSLAM%20Causal%20Bayesian%20Network%20Structure%20Learning%20of%20Mixed%20Type%20Data/_dagslam/DAGSLAM.py
#DAGSLAM is an extension of the NOTEARS algorithm developed by Xun Zheng, et al.


#helper function to infer loss types and generate m_vec for dataframe
"""
IN: df: DataFrame containing data
OUT: loss_type: array containing types of each variable logistic, gauss, or multi-logistic
    m_vec: stores total number of categories for each multinomial variable (has dim = number of columns)
"""
def infer_type(df):
    loss_type=[]
    m_vec=[]

    for col in df.columns:
        x=df[col].dropna()
        unique_vals=x.unique()
        num_unique = len(unique_vals)

        #infer variable type by combination of dtype and cardinality
        if np.issubdtype(x.dtype, np.number):
            #numeric datatypes
            if num_unique == 2 and set(unique_vals).issubset({0,1}): #binary numeric variable
                loss_type.append("logistic")
                m_vec.append(1) # 1 "category"
            else: #continuous 
                loss_type.append("gauss")
                m_vec.append(1)
        else: #non-numeric
            if num_unique == 2:
                #binary categorical 
                loss_type.append("logistic")
                m_vec.append(1)
            else:
                #multi-class categorical
                loss_type.append("multi-logistic")
                m_vec.append(num_unique)

    return loss_type, m_vec

#Function to run dagslam on some data and output estimated DAG
"""
IN: df: pandas dataframe containing the data to be learned on
    lambda1: L1 regularisation parameter higher lambda leads to sparser DAGs
    max_iter: maximum number of iterations in dual ascent 
    w_threshold: sensitivity parameter, higher w_threshold makes the model less sensitive (set at default values)
OUT: W_est: Binary adjacency matrix of a DAG
"""
def run_dagslam(df, lambda1=0.03, max_iter=100, w_threshold=0.25): #changed lambda 1 0.03 ->0.1 w_threshold 0.25->0.3
    #guardrail for n/a's
    df_clean = df.dropna().copy()
    #encode non numeric values appropriately
    df_enc = encode_mixed_df(df_clean)

    #get loss types
    loss_type, m_vec =infer_type(df_clean)
    assert list(df_clean.columns) == list(df_enc.columns)
    X=df_enc.to_numpy(dtype=float)

    #Run DAGSLAM and its run time
    start = time.time() 
    W_est = dagslam(X, loss_type=loss_type, m_vec=m_vec, lambda1=lambda1,max_iter=max_iter, w_threshold=w_threshold)
    end = time.time()
    print(f"DAGSLAM took {(end - start)/60:.2f} minutes")

    return W_est

#draw graphs based on DAGSLAM weighted adjacency matrix
"""
IN: W: A weighted adjacency matrix (for all intents and purposes, inputs to this are only ever binary)
    output_path: path to save image to
    node_labels: labels for nodes
    threshold: min threshold for plotting edges in weighted adjacency matrices
OUT: None.
"""
def draw_graph(W, output_path, node_labels=None,threshold=0):
    n = W.shape[0] # number of nodes
    G = nx.DiGraph() # initialise empty directed graph
    
    #initialise default node labels
    if node_labels is None:
        node_labels = [f"X{i}" for i in range(n)]

    #add nodes to digraph
    for i, name in enumerate(node_labels):
        G.add_node(i, label=name)

    #add edges for surviving weights (thresholding done during the dagslam phase)
    for i in range(n):
        for j in range(n): # for each possible edge 
            w=W[i,j]
            if abs(w)>threshold:
                G.add_edge(i, j, weight=w)

    plt.figure(figsize=(6,6))
    pos = nx.spring_layout(G, seed=0)
    nx.draw(G, pos, with_labels=True, labels={i: node_labels[i] for i in range(n)},
            node_size=800, font_size=8, arrowsize=10)
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()

#alt graph drawing function (preferred)
"""
IN: W: adjacency matrix weighted or not
    out_path: output path to save image
    node_labels: labels for nodes
    threshold: required threshold above which to plot edges
OUT: None.
"""
def draw_graphviz_dag(W, out_path, node_labels=None, threshold=0.0, engine="dot"):

    W = np.asarray(W)
    if W.ndim == 1:
        W = W.reshape(1, 1)

    n = W.shape[0]

    # Default labels use X0, X1,... if no node labels are given
    if node_labels is None:
        node_labels = [f"X{i}" for i in range(n)]
    else:
        node_labels = list(node_labels)[:n]

    out_path = Path(out_path)
    g = Digraph(format="png", engine=engine)

    g.attr(rankdir="LR")
    g.attr(
        "node",
        shape="ellipse",
        style="solid",
        color="black",
        fontname="Helvetica",
        fontsize="10",
    )
    g.attr(
        "edge",
        color="black",
        arrowsize="0.7",
    )

    # Add nodes
    for name in node_labels:
        g.node(name, label=name)

    # Add edges for weights above threshold
    for i, src in enumerate(node_labels):
        for j, tgt in enumerate(node_labels):
            w = W[i, j]
            if abs(w) > threshold:
                g.edge(src, tgt)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    g.render(filename=out_path.with_suffix("").as_posix(), cleanup=True)

In [31]:
#Utility functions for DAGBagM
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri, conversion

# load R package
ro.r('.libPaths(c("/dcs/23/u2200504/R/x86_64-redhat-linux-gnu-library/4.5", .libPaths()))')
ro.r('library(dagbagM)')

#assigns node types. 'c' for continuous, 'b' for binary
"""
IN: df: Dataframe containing data to classify
OUT: a string vector of node types
"""
def infer_type(df):
    import rpy2.robjects as ro
    node_types=[]
    for col in df.columns:
        x=df[col].dropna()
        unique_vals=x.unique()
        if np.issubdtype(x.dtype, np.number):
            if len(unique_vals) == 2 and set(unique_vals).issubset({0, 1}):
                node_types.append("b")
            else:
                node_types.append("c")
        else:
            if len(unique_vals) == 2:
                node_types.append("b")
            else:
                node_types.append("c")
    return ro.StrVector(node_types)

"""
IN: df: DataFrame of data to learn on
    seed: seed for ``randomness"
OUT: np adjacency array
"""
def run_dagbagm(df: pd.DataFrame, seed=1):
    df_clean = df.dropna().copy()
    #infer node types on transformed df
    node_type = infer_type(df_clean)

    print("Columns used in dagbagM:", df_clean.columns.tolist())
    print("dtypes:", df_clean.dtypes)

    with (ro.default_converter + pandas2ri.converter).context():
        #Python ->R
        Y_r = conversion.py2rpy(df_clean)

        ro.globalenv["Y"] = Y_r
        ro.globalenv["node_type"] = node_type
        ro.globalenv["seed"] = seed
        ro.r('''
        set.seed(seed)
        Y_df <- as.data.frame(Y)

        # Convert to numeric matrix for hc
        Y_mat <- as.matrix(Y_df)
        
        temp <- dagbagM::hc(
          Y = Y_mat,
          nodeType = node_type,
          whiteList = NULL,
          blackList = NULL,
          tol = 1e-6,
          standardize = FALSE,
          maxStep = 1000,
          restart = 10,
          verbose = FALSE
        )
        adj_mat <- temp$adjacency
        col_names <- colnames(Y_mat)
        ''')
        #R -> Python
        adjacency = conversion.rpy2py(ro.r('adj_mat'))
        col_names= list(ro.r('col_names'))

    return np.asarray(adjacency), col_names

In [4]:
#Utility functions for HC
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri, conversion

# load R package
ro.r('.libPaths(c("/dcs/23/u2200504/R/x86_64-redhat-linux-gnu-library/4.5", .libPaths()))')
ro.r('library(bnlearn)')

#run hc function
"""
IN: df: Dataframe containing data to be learned on
    seed: seed to initialise environment
OUT: adj: np adjacency matrix
    nodes: node labels
"""
def run_hc(df, seed=1):
    #remove NA rows
    df_clean = df.dropna().copy()
    with (ro.default_converter+pandas2ri.converter).context():
        ro.globalenv["df"]=conversion.py2rpy(df_clean)
        ro.globalenv["seed"]= seed

        ro.r('''
        library(bnlearn)
        data_df <- as.data.frame(df)
        
        any_disc <- FALSE
        any_cont <- FALSE

        for (i in seq_along(data_df)) {
          col <- data_df[[i]]
          #go through numeric column and flag whether there are discrete or continuous values
          if (is.numeric(col)) {
            vals <- unique(col[!is.na(col)])
            if (length(vals) <= 6 && all(abs(vals - round(vals)) < 1e-8)) {
              data_df[[i]] <- factor(col)
              any_disc <- TRUE
            } else {
              data_df[[i]] <- as.numeric(col)
              any_cont <- TRUE
            }
          } else {
            data_df[[i]] <- factor(col)
            any_disc <- TRUE
          }
        }

        #detect if there are mixed types, continuous only, or discrete only
        if (any_disc && any_cont) {
          chosen_score <- "bic-cg"    # mixed conditional Gaussian
        } else if (any_disc && !any_cont) {
          chosen_score <- "bde"       # discrete-only
        } else if (!any_disc && any_cont) {
          chosen_score <- "bic-g"     # Gaussian-only
        } else {
          stop("No usable variables (neither discrete nor continuous).")
        }
        
        set.seed(seed)

        #run hill climbing with conditional score
        dag_hc <- hc(data_df, score=chosen_score)

        adj <- amat(dag_hc)
        nodes <- colnames(adj)
        ''')
        
        #convert adjacency matrix in R back to python
        adj = conversion.rpy2py(ro.r("adj"))
        nodes=list(ro.r("nodes"))

    #coerce adjacency matrix into numeric 2D array
    adj=np.asarray(adj, dtype=int)
    return adj,nodes

In [34]:
#Utility functions for LiM
import lingam

"""LiM expects a numeric matrix with continuous and discrete columns as well as
an array per variable indicating 0 for discrete and 1 for continuous.
Similar to what we did for DAGBagM, we write a helper function to infer the datatype."""

def infer_type(df):
    """
    Infers a 'flag array' for a given dataframe.
    0 corresponds to discrete variables, 1 to continuous variables
    """

    flags=[]
    for col in df.columns:
        x = df[col].dropna()
        #classify numeric columns
        if np.issubdtype(x.dtype, np.number):
            vals = x.unique()
            if len(vals) <= 1:
                flags.append(0) #variable is discrete
            else:
                flags.append(1) # variable is continuous
    return np.asarray(flags, dtype=int)

def run_lim(df, seed=1):
    """
    Clean dataframe and generate the numeric matrix as well as flag array.
    Run LiM on the data.
    """

    df_clean=df.dropna().copy()
    flags = infer_type(df_clean)
    flags_2d = flags.reshape(1,-1)
    cols = list(df_clean.columns)

    X = df_clean.to_numpy(dtype=float)

    start = time.time()
    model = lingam.LiM()

    model.fit(X, flags_2d, only_global=True)
    #LiM creates an adjacency matrix
    M = model.adjacency_matrix_
    #coerce adjacency matrix into a matrix of 0's and 1's
    adj = (np.abs(M) > 0).astype(int)
    end=time.time()

    print(f"LiM took {(end - start)/60:.2f} minutes")
    return adj, cols

In [ ]:
#Utility functions for ReX
from causalexplain import GraphDiscovery
import re

#function to convert a dot object to a usual adjacency matrix
"""
IN: dot_path: path to file containing dot file
OUT: adj:adjacency matrix
    nodes: list of node labels
"""
def dot_to_adjacency(dot_path):
    G = nx.drawing.nx_pydot.read_dot(str(dot_path))
    nodes = list(G.nodes())
    idx = {n: i for i, n in enumerate(nodes)}
    p = len(nodes)
    adj = np.zeros((p, p), dtype=int)
    for u, v in G.edges():
        i = idx[u]
        j = idx[v]
        adj[i, j] = 1

    adj = np.asarray(adj)
    if adj.ndim == 1:              # safety
        adj = adj.reshape(1, 1)
    return adj, nodes

#makes column names ReX friendly i.e start with letter, contain only letters and numbers
"""
IN: df: df containing data with labelled variables
OUT: df2: cleaned df with ne ReX friendly column labels
"""
def clean_name(df):
    new_cols = []
    for c in df.columns:
        # remove underscores and other non-alphanumeric
        c2 = re.sub(r'[^A-Za-z0-9]', '', c)
        # if it doesn't start with a letter, prefix with 'X'
        if not c2 or not c2[0].isalpha():
            c2 = "X" + c2
        new_cols.append(c2)
    df2 = df.copy()
    df2.columns = new_cols
    return df2

#runs the ReX causal algorithm
"""
IN: df: dataframe containing data to learn on
OUT: adj: np adjacency matrix
    nodes: node labels
"""
def run_rex(df, experiment_name="rex_exp"):
    #Graph Discovery expects a file path for dataframe
    tmp_csv = "tmp_rex_input.csv"
    df = clean_name(df)
    df.to_csv(tmp_csv, index=False) #write df to a csv temporarily

    #init graph discovery instance
    start = time.time()
    gd = GraphDiscovery(experiment_name=experiment_name, model_type="rex",csv_filename= tmp_csv)

    gd.run(quiet=True)
    end=time.time()

    dot_path = output_dir / f"{experiment_name}.dot"
    gd.export_dag(str(dot_path))

    # Read DOT with networkx and build adjacency
    adj, nodes = dot_to_adjacency(dot_path)
    print("ReX adj ndim:", adj.ndim, "shape:", adj.shape)
    print(f"ReX took {(end - start)/60:.2f} minutes")

    return adj, nodes

In [9]:
#Causal Pitfalls Skeleton Code (adapted and synthesised from CP notebooks)
# loop over datasets
csv_files = sorted(cp_root.rglob("*.csv"))
results = []

for csv_path in csv_files:
    #relative path
    rel = csv_path.relative_to(cp_root)

    #we only examine those with known ground truths i.e with _truth
    if csv_path.stem.endswith("_truth"):
        continue

    #expected ground truth path
    truth_path = csv_path.with_name(csv_path.stem + "_truth.csv")
    if not truth_path.exists():
        print("  Skipped (no ground-truth file)")
        continue

    print(f"Processing: {rel}")

    try:
        df = pd.read_csv(csv_path)
        if df.empty:
            print("  Skipped (empty file)")
            continue

        # Run FCI on this dataset
        data = df.to_numpy(dtype=float)
        #REPLACE WITH APPROPRIATE CD ALGO
        g, _ = fci(data, fisherz, alpha=0.05) #causal graph object

        #extract adjacency matrix from PDAG
        W_pdag = g.graph
        print(W_pdag)

        #Output schema scenario__file__fci.png
        parts = rel.parts           
        scenario = parts[0] if len(parts) > 1 else "root"
        name_no_ext = csv_path.stem

        out_name = f"{scenario}__{name_no_ext}__FCI.png"
        out_path = output_dir / out_name

        # Save PNG using GraphUtils
        py_dot = GraphUtils.to_pydot(g, labels=list(df.columns))
        py_dot.write_png(str(out_path))

        #load ground truth and score
        gt_df = pd.read_csv(truth_path)
        gt_df.index.name = None
        gt_df.columns.name = None

        W_true = gt_df.to_numpy()
            
        # compute scores
        metrics = score_pdag(W_pdag, W_true, labels_pdag=list(df.columns), labels_true=list(gt_df.columns))
        print("metrics computed")
        print(metrics)
        metrics.update(
            dict(
                scenario=scenario,
                dataset=name_no_ext,
                algo="fci"
            )
        )
        results.append(metrics)
        
    except Exception as e:
        print(f"  ERROR on {rel}: {e}")

scores_df = pd.DataFrame(results)

#Store computed scores for the experiment
scores_path = project_root / "results" / "scores"/"scores_dagslam.csv"#REPLACE WITH CORRECT PATH
scores_path.parent.mkdir(parents=True, exist_ok=True)
scores_df.to_csv(scores_path, index=False)

  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
Processing: casual_effect/device_failure_data.csv
  ERROR on casual_effect/device_failure_data.csv: name 'fci' is not defined
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
Processing: casual_effect/student_tutoring_data.csv
  ERROR on casual_effect/student_tutoring_data.csv: name 'fci' is not defined
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no grou

In [37]:
#NIJ Skeleton code (adapted and synthesised from NIJ notebooks)
csv_path = nij_root/"NIJ_lean_compact_onehot.csv"
df = pd.read_csv(csv_path)

# Run *INSERT CAUSAL DISCOVERY ALGORITHM* on this dataset
#df = encode_mixed_df(df) sometimes it is required to encode non numerical data with categorical variables
W_est = run_dagslam(df) #Replace as necessary

#save recovered adjacency matrix to a csv
adj_df = pd.DataFrame(W_est, index=df.columns, columns=df.columns)
adj_out_path = output_dir / "NIJ_graph_DAGSLAM_adj.csv"
adj_df.to_csv(adj_out_path, index=True)

out_path = output_dir / "NIJ_graph_DAGSLAM"

#Post-hoc processing
#convert the adjacency array into a networkX digraph for easier processing
W_est = np.asarray(W_est)
if W_est.ndim == 1:
    W_est = W_est.reshape(1, 1)

n = W_est.shape[0] # number of nodes
G = nx.DiGraph() # initialise empty directed graph
node_labels=list(df.columns)

for name in node_labels:
    G.add_node(name)

# add edges using string IDs
n = W_est.shape[0]
for i, src in enumerate(node_labels):
    for j, tgt in enumerate(node_labels):
        w = W_est[i, j]
        if abs(w) > 0:
            G.add_edge(src, tgt, weight=w)

AGE_PREFIX = "Age_at_Release_"          # prefix of one-hot age bucket columns
AGE_MERGED = "Age_at_Release"           # name of merged age node
TARGET     = "Recidivism_Within_3years" #target node
#G assumed to be an nx.DiGraph()

#Merge age buckets
age_nodes = [n for n in G.nodes if isinstance(n, str) and n.startswith(AGE_PREFIX)]
G_merged = nx.DiGraph()

# copy all nodes except age buckets
for n in G.nodes:
    if n not in age_nodes:
        G_merged.add_node(n)

# add merged age node if any buckets exist
if age_nodes:
    G_merged.add_node(AGE_MERGED)

age_out = Counter()   #counts outgoing edges from age buckets Age -> X (bucket -> non-age)
age_in  = Counter()   #counts incoming edges ".. "(non-age -> bucket)

for u, v, data in G.edges(data=True):

    #ignore edges between age buckets
    if u in age_nodes and v in age_nodes:
        continue

    #edges not affecting age buckets are just copied over
    if u not in age_nodes and v not in age_nodes:
        G_merged.add_edge(u, v, **data)
        continue

    #bucket -> non-age variable
    if u in age_nodes and v not in age_nodes:
        age_out[v] += 1

    #non-age variable -> bucket
    if v in age_nodes and u not in age_nodes:
        age_in[u] += 1

#decide majority direction for each neighbour of an age bucket
for x, cnt_out in age_out.items():
    cnt_in = age_in.get(x, 0)

    if cnt_out > cnt_in:
        # majority oriented as Age bucket -> X
        G_merged.add_edge(AGE_MERGED, x)
    elif cnt_in > cnt_out:
        # majority X -> Age bucket
        G_merged.add_edge(x, AGE_MERGED)
    else:
        #tie: drop, dont add edge to age merged
        pass

# add edges X -> Age for neighbours that only ever had incoming-to-bucket edges
for x, cnt_in in age_in.items():
    if x in age_out:
        continue    # already handled above
    G_merged.add_edge(x, AGE_MERGED)

if TARGET not in G_merged:
    raise ValueError(f"Target node '{TARGET}' not found in graph; "
                     f"available nodes include: {list(G_merged.nodes)[:10]} ...")

#choose nodes to keep
parents  = set(G_merged.predecessors(TARGET))
markov_blanket_nodes = parents | {TARGET}

#prune the graph
G_mb = G_merged.subgraph(markov_blanket_nodes).copy()

print("Original nodes:", len(G.nodes))
print("After age-merge:", len(G_merged.nodes))
print("Markov blanket nodes:", len(G_mb.nodes))

print("Original edges:", len(G.edges()))
print("After age-merge:", len(G_merged.edges()))
print("Markov blanket edges:", len(G_mb.edges()))

node_labels = list(G_mb.nodes())          

#save post-hoc processed graph
out_path = output_dir / "NIJ_graph_DAGSLAM_PRUNED"
draw_graphviz_from_nx(G_mb, out_path)

#read estimated adjacency matrix back in
csv_path = output_dir/ "NIJ_graph_DAGSLAM_adj.csv"
adj_df = pd.read_csv(csv_path, index_col=0)
node_labels = list(adj_df.index)
#convert to a numpy array
A = adj_df.to_numpy()

seeds = [42,7,12]
incompat_scores_seed = []
disagree_seed = []
#run experiments across seeds
for seed in seeds:
    score = incompatibility_score(A, k = 5, n_subsets=50, seed=seed)
    incompat_scores_seed.append(score)
    disagree_seed.append(goodness(score, 5))

print(str(incompat_scores_seed))
print(str(disagree_seed))

#run experiments across subset sizes
sizes = [5,10,15]
incompat_scores = []
disagree = []
for size in sizes:
    score = incompatibility_score(A, k = size, n_subsets=50, seed=42)
    incompat_scores.append(score)
    disagree.append(goodness(score, size))

print(str(incompat_scores))
print(str(disagree))

#run edge stability experiment
edge_freq_dagslam = bootstrap_edge_stability(df_full, B=20, seed=42)

#save edge stability result to CSV
edge_freq_df = pd.DataFrame(edge_freq_dagslam, index=node_labels, columns=node_labels)
edge_freq_df.to_csv(output_dir / "NIJ_DAGSLAM_edge_stability.csv")
print("Number of edges with freq >= 0.5:",
      np.sum(edge_freq_dagslam >= 0.5))
print("Number of edges with freq >= 0.8:",
      np.sum(edge_freq_dagslam >= 0.8))

iter:0
rho:1.0
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=6834.433221122876
loss=9803.13604777547
loss=6834.360036315092
loss=6834.300297068414
loss=6834.081795152393
loss=6832.074701733789
loss=6826.4545571030285
loss=6813.090322271924
loss=6793.048937971541
loss=6789.254276315549
loss=6787.387754589591
loss=6783.315974983922
loss=6777.287651782135
loss=6773.108720973831
loss=6769.055426922938
loss=6765.779800188809
loss=6765.230859336954
loss=6759.282312228648
loss=6760.875502739124
loss=6757.771831161692


KeyboardInterrupt: 